<img src=https://audiovisuales.icesi.edu.co/assets/custom/images/ICESI_logo_prin_descriptor_RGB_POSITIVO_0924.jpg width=200>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanestebancg2806/sugarcane-yield-quality-prediction/blob/main/03_comparacion_modelos_ipsa.ipynb)

# **Proyecto Caña de Azúcar**
## Modelado y generación de insights para el TCH y la sacarosa en el Ingenio Providencia


--------------------------------------------------------------------

**Aprendizaje Automático II - Proyecto final**

**Profesor:** Anibal Sosa

**Integrantes:**

*   Juan David Martínez Legarda
*   Daniel Velasco López
*   Juan Esteban Cardona García
*   Katherin Adriana Camargo Cetina

Fecha: 22/11/2026

________________________________________________________________________

# IPSA (CC01-1940): comparación de modelos de clasificación

En este notebook se construyen y comparan modelos para clasificar cada cosecha de la variedad CC01-1940 en un nivel bajo, medio o alto de tonelaje (`TCH`, toneladas de caña por hectárea) y de sacarosa (porcentaje de sacarosa en la caña). Los dos targets se tratan como problemas independientes.

**Escenario de uso.** Los modelos clasifican una cosecha en el momento de programar su corte. En ese momento se conocen el número de corte, la dosis de madurante y el grupo de tenencia, y, como la fecha de corte ya está programada, también la edad del cultivo, las semanas desde la aplicación del madurante y el mes de cosecha. La lluvia registrada se acumula hasta la cosecha con una ventana que no está documentada, así que solo se conoce completa cerca del corte. Por eso los modelos describen la programación del corte y no un pronóstico con meses de anticipación. Las variables que se miden en la cosecha o en el molino quedan fuera de los predictores.

El recorrido va de lo simple a lo complejo: dos clasificadores sin predictores como piso, la regresión logística multinomial con regularización y KNN como modelos de referencia, y Random Forest, XGBoost y máquinas de vectores de soporte (SVM) como modelos avanzados. Después se prueban dos alternativas: agregar el resultado de la cosecha anterior de cada suerte, y predecir el valor continuo de cada target con un modelo de regresión para asignar la clase con los umbrales del Ingenio.

Para que la comparación sea justa, todos los modelos comparten el preprocesamiento, la partición en entrenamiento y prueba, los pliegues de validación cruzada y el criterio de selección. Así, una diferencia de desempeño se puede atribuir al modelo y no a la forma de evaluarlo.

**Criterios generales**

- La partición 80/20 se hace al inicio y se agrupa por periodo de cosecha: ningún mes de cosecha aparece a la vez en entrenamiento y en prueba.
- Los hiperparámetros y la selección de modelos se deciden con validación cruzada sobre el entrenamiento, con la misma agrupación: 5 pliegues repetidos con 3 asignaciones distintas de los periodos, para un total de 15 pliegues fijos y comunes a todos los modelos.
- La imputación, la codificación y el escalado se estiman dentro de un `Pipeline`, de modo que en cada pliegue solo usan los datos de entrenamiento de ese pliegue.
- La regla para elegir el mejor modelo se fija antes de comparar los modelos.
- El conjunto de prueba se usa una sola vez, al final, con todos los modelos ya elegidos.

**Estructura**

1. Carga del dataset procesado
2. Preprocesamiento
3. Partición, pliegues de validación cruzada y auditoría de fuga de información
4. Métricas y regla de elección del modelo
5. Modelos de referencia: clasificadores sin predictores, regresión logística multinomial y KNN
6. Modelos avanzados: Random Forest, XGBoost, SVM, historial de la suerte y regresión con umbrales
7. Comparación en validación cruzada y elección del modelo
8. Evaluación en el conjunto de prueba
9. Interpretabilidad
10. Conclusiones

## Punto de partida: qué dejó el análisis exploratorio

El EDA y la preparación de los datos se hicieron en `01_eda_ipsa.ipynb`. A continuación se resume lo que se necesita de ese trabajo para el modelado.

**Los datos.** `BD_IPSA_1940.xlsx` es un subconjunto del Ingenio Providencia con 2187 cosechas entre julio de 2014 y enero de 2021. Todas comparten tres condiciones: la variedad CC01-1940, la maduración química con BONUS 250 EC y la cosecha mecanizada en verde. Las conclusiones aplican solo a suertes con esas características. Cada fila es una suerte (identificada por la hacienda `FAZ` y la suerte `TAL`) cosechada en un mes (`periodo`, en formato AAAAMM). El dataset tiene 1115 suertes de 285 haciendas, y 629 de esas suertes aparecen en dos o más cosechas, así que los registros no son independientes entre sí.

**Las clases.** Cada target se divide en tres niveles con los umbrales del Ingenio:

| Clase | `TCH` (t/ha) | Sacarosa (%) |
|---|---|---|
| Bajo | menos de 125 | menos de 12.2 |
| Medio | de 125 a 150 | de 12.2 a 12.9 |
| Alto | más de 150 | desde 13.0 |

Los umbrales son valores fijos que no dependen de los datos, así que construir las clases no filtra información del conjunto de prueba. El umbral inferior coincide con el percentil 25 de cada target, y el superior queda cerca del percentil 60. La clase baja, que agrupa las cosechas que el Ingenio necesita detectar para intervenir, es la más pequeña. Los 4 registros con `TCH` menor que 60 t/ha se consideraron errores de registro: no tienen clase de `TCH`, pero conservan la de sacarosa. Los dos targets tienen una correlación débil (Spearman de −0.21), así que se modelan como dos problemas independientes.

**Los predictores y su justificación.** El modelo principal usa 7 variables originales, comunes a los dos targets:

| Predictor | Qué mide | Por qué entra | Tratamiento |
|---|---|---|---|
| `edad` | Edad del cultivo en la cosecha (meses) | Más edad, más biomasa acumulada; en sacarosa la relación tiene forma de U invertida | Sin nulos |
| `cortes` | Número de corte de la cepa | El tonelaje baja con cada corte sucesivo | Sin nulos |
| `semsmad` | Semanas entre la aplicación del madurante y el corte | La sacarosa sube hasta unas 12 semanas y luego se estabiliza | 2 valores negativos tratados como faltantes |
| `dosismad` | Dosis de madurante (L/ha) | Fundamento agronómico. En el EDA no mostró relación con los targets, pero el 98 % de las dosis está entre 0.5 y 1.5 L/ha, un rango demasiado estrecho para descartar su efecto | 2 valores inválidos (0.0 y 9.0) tratados como faltantes |
| `lluvias` | Lluvia registrada (mm), con ventana de acumulación no documentada | Principal predictor numérico de la sacarosa (Spearman de −0.28) | El 31.3 % son ceros que corresponden a falta de medición; se imputan con la mediana del mes de cosecha y se acotan en el percentil 99 |
| `mes` | Mes de cosecha | La sacarosa tiene dos máximos al año, tras cada temporada seca del valle del río Cauca [2] | Dos armónicos (periodos de 12 y 6 meses), que capturan el 93 % de la variación mensual de la sacarosa |
| `grupo_tenencia` | Grupo de tenencia de la hacienda (códigos 1, 2 y 3 sin significado documentado) | Relación débil con `TCH` | Variables indicadoras, con el código 1 como referencia |

Con los cuatro armónicos del mes y las dos indicadoras de la tenencia, las 7 variables generan 11 columnas en la matriz de diseño. En el EDA, ningún predictor numérico candidato tuvo un VIF mayor que 5. El VIF de la matriz de diseño completa, que incluye los armónicos y las indicadoras, se calcula con el conjunto de entrenamiento en la sección 3.

**Lo que quedó fuera de los predictores**

| Columnas | Motivo |
|---|---|
| `NOME`, `FAZ`, `TAL` | Identifican la cosecha, no describen el cultivo. `FAZ` y `TAL` se usan para identificar cada suerte |
| `variedad`, `madurada`, `producto`, `tipocorte` | Son constantes en todo el dataset |
| `vejez` (horas entre el corte y el molino) y `me` (materia extraña, con confianza moderada) | Se miden al recibir la caña en el molino, después del corte. Usarlas sería fuga de información |
| `pct_diatrea` (porcentaje de entrenudos barrenados por *Diatraea*) | Según la metodología de Cenicaña, se mide en la cosecha, muestreando 100 tallos por suerte y contando los entrenudos barrenados [1]. Los datos no muestran que el Ingenio la estime antes del corte, así que no está disponible en el escenario de uso. Su aporte se evalúa como prueba de robustez en la sección 7 |
| `TCH` y sacarosa como predictores del otro target | Los dos se conocen solo después de la cosecha |
| `periodo` | Se usa para agrupar la partición y la validación cruzada, no como predictor. Su parte estacional entra por `mes` |
| `anio` | En el EDA, su relación con `TCH` la dominaba un solo año (2017, el de mayor lluvia), y un modelo aplicado a cosechas futuras no puede extrapolar el efecto de años que no vio. En el análisis de regresión del histórico de suertes, además, el efecto del año reflejaba condiciones propias de cada periodo, comunes a todas las fincas, que no se trasladan a una zafra nueva. Su aporte se evalúa como prueba de robustez en la sección 7 |
| `lluvia_sin_dato` | En el EDA, los registros sin lluvia medida no difieren de los demás en ninguno de los dos targets: su información mutua con ambos quedó por debajo del nivel de ruido |

**Relaciones que encontró el EDA**

- Todas las relaciones con los targets son débiles: ningún predictor numérico supera una correlación de Spearman de 0.29 en valor absoluto. Se espera un poder predictivo limitado.
- El tonelaje se relaciona sobre todo con el número de corte (Spearman de −0.27) y con la edad del cultivo (0.15), y no tiene estacionalidad: sus medianas mensuales van de 141 a 152 t/ha.
- La sacarosa depende sobre todo de la época de cosecha: el mes explica el 21 % de su variación, con medianas más altas en marzo, septiembre y octubre (13.2 % a 13.4 %) y más bajas entre abril y julio (12.1 % a 12.5 %). Más lluvia se asocia con menos sacarosa.
- Dos relaciones con la sacarosa no son lineales: con la edad tiene forma de U invertida (sube de 12.5 % a cerca de 13.0 % hacia los 12.5 meses y baja después), y con las semanas de madurante crece hasta unas 12 semanas y luego se estabiliza. Un modelo lineal en esas variables no las captura por completo.

**Limitaciones de los datos**

- `lluvias` es un único valor por registro, con una ventana de acumulación no documentada, y casi un tercio de sus valores no tiene medición.
- El significado de `me` y de los códigos de `grupo_tenencia` no está confirmado por el Ingenio, y el de `semsmad` se dedujo de los datos.
- Parte de la variación de la sacarosa depende de factores posteriores al corte (materia extraña y horas hasta el molino), que no están disponibles antes de la cosecha.

In [1]:
# Librerías generales del notebook. Los módulos de scikit-learn y xgboost se importan en la sección donde se usan.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import xgboost

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

%matplotlib inline

# Versiones usadas, para la reproducibilidad de los resultados
print(f"pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__} | "
      f"xgboost {xgboost.__version__}")

pandas 3.0.6 | numpy 2.5.4 | scikit-learn 1.9.1 | xgboost 3.4.1


## 1. Carga del dataset procesado

La última celda de `01_eda_ipsa.ipynb` guarda el dataset en `datos/ipsa_limpio.parquet`. El archivo trae solo transformaciones que dan el mismo resultado en cada fila sin importar las demás: los valores inválidos de `lluvias`, `dosismad` y `semsmad` convertidos en faltantes, la marca `excluir_tch` y las clases con los umbrales del Ingenio. No trae ninguna imputación, porque los estadísticos para imputar deben calcularse solo con el conjunto de entrenamiento.

Antes de modelar se verifica que el archivo conserve lo establecido en el EDA:

- 2187 cosechas y 22 columnas, sin repetir la combinación de hacienda, suerte y periodo.
- Los faltantes de los predictores: 685 en `lluvias`, 2 en `semsmad` y 2 en `dosismad`. Si alguno cambia, el archivo llegó imputado.
- 4 registros con `excluir_tch`, sin clase de `TCH`, y clase de sacarosa en todos los registros.
- Que cada clase se pueda reconstruir con los umbrales del Ingenio a partir de su propio target. Así se confirma que las clases no dependen de ningún estadístico del dataset.

Además, las clases se codifican como enteros ordenados (bajo = 0, medio = 1, alto = 2), porque XGBoost no acepta etiquetas de texto. Es un mapeo fijo, sin estadísticos. Los armónicos del mes que trae el archivo no se usan: se recalculan dentro del pipeline a partir de `mes` (sección 2). `pct_diatrea` y `anio` se conservan en el DataFrame solo para las pruebas de robustez de la sección 7. Si alguna verificación falla, el notebook se detiene.

In [2]:
# 1. Carga del dataset procesado (salida de 01_eda_ipsa.ipynb) y verificación de sus propiedades
RANDOM_STATE = 42  # semilla común a la partición, los pliegues y los modelos
RUTA_IPSA = Path("datos/ipsa_limpio.parquet")

# Propiedades del archivo establecidas en 01_eda_ipsa.ipynb
FILAS_ESPERADAS, COLUMNAS_ESPERADAS = 2187, 22
NULOS_ESPERADOS = {"lluvias": 685, "semsmad": 2, "dosismad": 2}
FILAS_EXCLUIR_TCH = 4

# Predictores del modelo principal en su forma original. mes entra como número: sus armónicos se calculan en el pipeline
PREDICTORES_NUMERICOS = ["edad", "cortes", "semsmad", "lluvias", "dosismad"]
PREDICTOR_MES = "mes"
PREDICTORES_CATEGORICOS = ["grupo_tenencia"]
PREDICTORES = PREDICTORES_NUMERICOS + [PREDICTOR_MES] + PREDICTORES_CATEGORICOS

# Variables excluidas del modelo principal cuyo aporte se evalúa como robustez en la sección 7
PREDICTORES_ROBUSTEZ = ["pct_diatrea", "anio"]

# Columnas del archivo que no entran a ningún modelo (lluvia_sin_dato) o que se recalculan en el pipeline (armónicos)
COLUMNAS_NO_USADAS = ["lluvia_sin_dato", "mes_sin_12", "mes_cos_12", "mes_sin_6", "mes_cos_6"]

# Clases ordenadas, columna de clase de cada target y umbrales del Ingenio
CLASES = ["bajo", "medio", "alto"]
COLUMNA_CLASE = {"TCH": "clase_tch_ingenio", "sacarosa": "clase_sac_ingenio"}
COLUMNA_CODIGO = {"TCH": "codigo_tch", "sacarosa": "codigo_sac"}  # clase como entero: bajo 0, medio 1, alto 2
UMBRALES_INGENIO = {"TCH": (125, 150), "sacarosa": (12.2, 13.0)}


def formato_miles(numero):
    """Entero con espacio como separador de miles desde cinco cifras (norma RAE)."""
    return f"{numero:,}".replace(",", " ") if abs(numero) >= 10_000 else str(numero)


df_ipsa = pd.read_parquet(RUTA_IPSA)

# Estructura del archivo
assert df_ipsa.shape == (FILAS_ESPERADAS, COLUMNAS_ESPERADAS), f"Dimensiones inesperadas: {df_ipsa.shape}"
columnas_requeridas = PREDICTORES + PREDICTORES_ROBUSTEZ + COLUMNAS_NO_USADAS + list(COLUMNA_CLASE.values())
faltan = set(columnas_requeridas) - set(df_ipsa.columns)
assert not faltan, f"Faltan columnas en el archivo: {faltan}"
assert not df_ipsa.duplicated(subset=["FAZ", "TAL", "periodo"]).any(), "Hay cosechas repetidas"

# Faltantes originales: si cambian, el archivo llegó imputado
nulos = df_ipsa[PREDICTORES + PREDICTORES_ROBUSTEZ].isna().sum()
assert nulos[nulos > 0].to_dict() == NULOS_ESPERADOS, f"Faltantes inesperados: {nulos[nulos > 0].to_dict()}"

# Registros excluidos de TCH y completitud de las clases
assert df_ipsa["excluir_tch"].sum() == FILAS_EXCLUIR_TCH, "El número de registros con excluir_tch no es 4"
assert df_ipsa.loc[df_ipsa["excluir_tch"], "clase_tch_ingenio"].isna().all(), "Registros excluidos con clase de TCH"
assert df_ipsa.loc[~df_ipsa["excluir_tch"], "clase_tch_ingenio"].notna().all(), "Registros válidos sin clase de TCH"
assert df_ipsa["clase_sac_ingenio"].notna().all(), "Registros sin clase de sacarosa"
for columna in COLUMNA_CLASE.values():
    assert list(df_ipsa[columna].cat.categories) == CLASES and df_ipsa[columna].cat.ordered, \
        f"{columna} perdió el orden bajo < medio < alto"

# Las clases se reconstruyen con los umbrales del Ingenio a partir de su propio target
# TCH se registra en enteros: bajo < 125, medio de 125 a 150, alto > 150
assert pd.api.types.is_integer_dtype(df_ipsa["TCH"]), "TCH ya no está en valores enteros"
validos_tch = ~df_ipsa["excluir_tch"]
bajo_tch, alto_tch = UMBRALES_INGENIO["TCH"]
clase_tch_reconstruida = pd.cut(df_ipsa.loc[validos_tch, "TCH"], bins=[-np.inf, bajo_tch - 1, alto_tch, np.inf],
                                labels=CLASES)
# Sacarosa se registra con un decimal: bajo < 12.2, medio de 12.2 a 12.9, alto >= 13.0
bajo_sac, alto_sac = UMBRALES_INGENIO["sacarosa"]
clase_sac_reconstruida = pd.cut(df_ipsa["sacarosa"], bins=[-np.inf, bajo_sac, alto_sac, np.inf],
                                labels=CLASES, right=False)
assert (clase_tch_reconstruida.astype(str) == df_ipsa.loc[validos_tch, "clase_tch_ingenio"].astype(str)).all(), \
    "La clase de TCH no corresponde a los umbrales del Ingenio"
assert (clase_sac_reconstruida.astype(str) == df_ipsa["clase_sac_ingenio"].astype(str)).all(), \
    "La clase de sacarosa no corresponde a los umbrales del Ingenio"

# Codificación entera de las clases (mapeo fijo, sin estadísticos). Sin clase -> faltante
for target, columna in COLUMNA_CLASE.items():
    df_ipsa[COLUMNA_CODIGO[target]] = (df_ipsa[columna].cat.codes.astype("Int64")
                                       .mask(df_ipsa[columna].isna()))

# Identificador de suerte (hacienda + suerte), necesario para el historial y la auditoría de fuga
df_ipsa["suerte"] = df_ipsa["FAZ"].astype(str) + "_" + df_ipsa["TAL"]

print(f"Cosechas: {len(df_ipsa)} | haciendas: {df_ipsa['FAZ'].nunique()} | suertes: {df_ipsa['suerte'].nunique()} | "
      f"periodos: {df_ipsa['periodo'].nunique()} (de {df_ipsa['periodo'].min()} a {df_ipsa['periodo'].max()})")
print(f"Predictores del modelo principal: {len(PREDICTORES)} variables originales ({', '.join(PREDICTORES)})")

# Balance de clases de cada target: registros (enteros) y porcentaje
conteos = pd.DataFrame({target: df_ipsa[columna].value_counts().reindex(CLASES)
                        for target, columna in COLUMNA_CLASE.items()})
conteos.loc["total"] = conteos.sum()
porcentajes = (conteos / conteos.loc["total"] * 100).round(1)
display(pd.concat({"registros": conteos, "%": porcentajes}, axis=1))

Cosechas: 2187 | haciendas: 285 | suertes: 1115 | periodos: 72 (de 201407 a 202101)
Predictores del modelo principal: 7 variables originales (edad, cortes, semsmad, lluvias, dosismad, mes, grupo_tenencia)


registros               %         
            TCH sacarosa    TCH sacarosa
bajo        494      471   22.6     21.5
medio       850      802   38.9     36.7
alto        839      914   38.4     41.8
total      2183     2187  100.0    100.0

### Conclusión de la sección 1

El archivo cumple todas las verificaciones: tiene 2187 cosechas y 22 columnas, sin cosechas repetidas, los faltantes de los predictores son los originales del EDA y las clases de los dos targets se reconstruyen exactamente con los umbrales del Ingenio a partir de su propio valor. La imputación queda por hacer dentro del modelado, y las clases no dependen de ningún estadístico del dataset.

Las cosechas provienen de 285 haciendas y 1115 suertes, en 72 periodos entre julio de 2014 y enero de 2021. Muchas suertes aparecen en varios periodos, y la partición tiene que tener en cuenta esa estructura (sección 3).

El modelo principal usa 7 variables originales. Las clases quedan con este balance:

| Clase | `TCH` | Sacarosa |
|---|---|---|
| Bajo | 494 (22.6 %) | 471 (21.5 %) |
| Medio | 850 (38.9 %) | 802 (36.7 %) |
| Alto | 839 (38.4 %) | 914 (41.8 %) |
| Total | 2183 | 2187 |

En los dos targets la clase baja es la más pequeña, con poco más de la mitad de registros que cada una de las otras dos. El desbalance es moderado, pero suficiente para que un modelo que nunca predice la clase baja obtenga una accuracy aceptable. Por eso la métrica que decide se fija en la sección 4, y la ponderación de clases se trata como un hiperparámetro.

## 2. Preprocesamiento

El preprocesamiento aplica las reglas de imputación y codificación definidas en el EDA. Se implementa con transformadores de scikit-learn dentro de un `Pipeline`, y no como pasos sueltos sobre el DataFrame, por dos razones:

- En la validación cruzada y en la búsqueda de hiperparámetros, el `Pipeline` se reajusta en cada pliegue, de modo que cada estadístico se calcula solo con los datos de ajuste de ese pliegue.
- El conjunto de prueba recibe exactamente las mismas transformaciones, con los valores aprendidos en el entrenamiento.

### 2.1 Reglas y transformadores

Las reglas son de dos tipos. Las **sin estado** dan el mismo resultado en cada fila sin importar las demás, así que se aplican igual en entrenamiento y en prueba. Las **estimadas en el ajuste** calculan un estadístico con los datos de ajuste.

| Columnas | Transformación | Tipo |
|---|---|---|
| `lluvias` (con `mes` como apoyo) | Los faltantes se reemplazan con la mediana de `lluvias` de su mes de cosecha, calculada con los valores observados. Si un mes no tiene valores observados en el ajuste, se usa la mediana general. Después, los valores se acotan en el percentil 99 de los observados, para que unos pocos registros extremos no dominen las distancias de KNN y SVM | Estimada |
| `edad`, `cortes`, `semsmad`, `dosismad` | Faltantes reemplazados con la mediana. Solo `semsmad` y `dosismad` tienen faltantes (2 cada una) | Estimada |
| `mes` | Seno y coseno con periodos de 12 y 6 meses: $\sin(2\pi m/12)$, $\cos(2\pi m/12)$, $\sin(2\pi m/6)$ y $\cos(2\pi m/6)$. Diciembre y enero quedan contiguos, y el segundo par permite representar los dos máximos anuales de la sacarosa | Sin estado |
| `grupo_tenencia` | Indicadoras de los códigos 2 y 3, con el código 1 como referencia. Las categorías se fijan de antemano, así que todos los pliegues producen las mismas columnas | Sin estado |
| Todas las columnas | Estandarización con la media y la desviación del ajuste | Estimada |

Cuatro decisiones adicionales:

- **`mes` no entra como número.** Solo llega al modelo a través de sus armónicos, y la imputación de `lluvias` lo usa para elegir la mediana. Los armónicos se calculan dentro del `Pipeline` a partir de `mes`, y no se toman del archivo del EDA, para que la importancia por permutación de la sección 9 mueva el mes completo, con sus cuatro columnas a la vez.
- **El escalado es opcional.** La regresión logística lo necesita para que la penalización trate por igual a todos los coeficientes, y KNN y SVM porque miden distancias. Los árboles de Random Forest y XGBoost dividen cada variable por umbrales, que no cambian con una transformación monótona, así que se ajustan sin escalar.
- **Las indicadoras también se escalan** cuando hay escalado, para que pesen igual que las variables continuas en las distancias de KNN y SVM y en la penalización de la logística.
- **El preprocesador admite variables numéricas adicionales,** que se imputan con la mediana. Se usa en las pruebas de robustez con `pct_diatrea` y `anio` (sección 7). El historial de la suerte tiene su propio transformador, que se define en la sección 6.

El preprocesador devuelve un `DataFrame` con los nombres de las columnas, que se necesitan para leer los coeficientes y las importancias en la sección 9.

Antes de la partición no existe un conjunto de entrenamiento, y ajustar el preprocesador con todos los datos usaría el conjunto de prueba. Por eso los transformadores se verifican aquí con un ejemplo pequeño construido a mano, cuyos resultados se pueden calcular sin el código. Las verificaciones sobre los datos reales (que no queden faltantes, el escalado y el VIF) se hacen con el conjunto de entrenamiento, en la sección 3. La única comprobación que usa todo el dataset es la de los armónicos, porque es una transformación sin estado: se comparan con los que calculó el EDA.

In [3]:
# 2.1 Transformadores propios: lluvias por mes de cosecha y armónicos del mes
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted


class ImputadorLluvias(BaseEstimator, TransformerMixin):
    """Imputa lluvias con la mediana de su mes de cosecha y acota los valores al percentil indicado.

    Recibe dos columnas en este orden: lluvias y mes. Devuelve solo lluvias.
    Las medianas y el tope se calculan con los valores observados del conjunto de ajuste.
    """

    def __init__(self, percentil_tope=99):
        self.percentil_tope = percentil_tope

    def fit(self, X, y=None):
        lluvias, mes = self._separar(X)
        observado = ~np.isnan(lluvias)
        self.mediana_mes_ = pd.Series(lluvias[observado]).groupby(mes[observado]).median().to_dict()
        self.mediana_global_ = float(np.median(lluvias[observado]))
        self.tope_ = float(np.percentile(lluvias[observado], self.percentil_tope))
        self.n_features_in_ = 2
        return self

    def transform(self, X):
        check_is_fitted(self, "tope_")
        lluvias, mes = self._separar(X)
        # Mediana del mes; si el mes no tuvo datos observados en el ajuste, mediana general
        mediana = pd.Series(mes).map(self.mediana_mes_).fillna(self.mediana_global_).to_numpy()
        lluvias = np.where(np.isnan(lluvias), mediana, lluvias)
        return np.minimum(lluvias, self.tope_).reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        return np.array(["lluvias"], dtype=object)

    @staticmethod
    def _separar(X):
        X = np.asarray(X, dtype=float)
        return X[:, 0], X[:, 1].astype(int)


class ArmonicosMes(BaseEstimator, TransformerMixin):
    """Seno y coseno del mes de cosecha con periodos de 12 y 6 meses. Transformación fija por fila: no aprende
    ningún estadístico, así que da el mismo resultado en cualquier pliegue."""

    PERIODOS = (12, 6)

    def fit(self, X, y=None):
        self.n_features_in_ = 1
        return self

    def transform(self, X):
        mes = np.asarray(X, dtype=float).reshape(-1)
        if not np.isin(mes, np.arange(1, 13)).all():
            raise ValueError("El mes de cosecha debe ser un entero entre 1 y 12")
        columnas = []
        for periodo in self.PERIODOS:
            angulo = 2 * np.pi * mes / periodo
            columnas += [np.sin(angulo), np.cos(angulo)]
        return np.column_stack(columnas)

    def get_feature_names_out(self, input_features=None):
        return np.array([f"mes_{funcion}_{periodo}" for periodo in self.PERIODOS for funcion in ("sin", "cos")],
                        dtype=object)


# Verificación de ImputadorLluvias con un ejemplo construido a mano
# Observados: mes 1 → 100; mes 2 → 300 y 2000; mes 12 → 150. El mes 3 no tiene valores observados.
ejemplo_lluvias = pd.DataFrame({"lluvias": [100, np.nan, 300, np.nan, 2000, 150],
                                "mes": [1, 1, 2, 3, 2, 12]})
imputador_ejemplo = ImputadorLluvias().fit(ejemplo_lluvias)
# Mediana general de los observados: (150 + 300) / 2 = 225. Percentil 99 (interpolación lineal): 300 + 0.97 × 1700 = 1949
assert imputador_ejemplo.mediana_mes_ == {1: 100, 2: 1150, 12: 150}, imputador_ejemplo.mediana_mes_
assert imputador_ejemplo.mediana_global_ == 225 and np.isclose(imputador_ejemplo.tope_, 1949)
lluvias_esperadas = [100, 100, 300, 225, 1949, 150]  # fila 2: mediana del mes 1; fila 4: mes sin datos → 225; fila 5: tope
assert np.allclose(imputador_ejemplo.transform(ejemplo_lluvias).ravel(), lluvias_esperadas)
print("ImputadorLluvias:", imputador_ejemplo.transform(ejemplo_lluvias).ravel().round(1).tolist())

# Verificación de ArmonicosMes: es una transformación sin estado, así que se compara con los armónicos del EDA
armonicos = ArmonicosMes().fit_transform(df_ipsa[[PREDICTOR_MES]])
columnas_armonicos = list(ArmonicosMes().get_feature_names_out())
assert columnas_armonicos == ["mes_sin_12", "mes_cos_12", "mes_sin_6", "mes_cos_6"]
assert np.allclose(armonicos, df_ipsa[columnas_armonicos].to_numpy()), "Los armónicos no coinciden con los del EDA"
print("ArmonicosMes: coincide con los armónicos del EDA en las", len(df_ipsa), "cosechas")

ImputadorLluvias: [100.0, 100.0, 300.0, 225.0, 1949.0, 150.0]
ArmonicosMes: coincide con los armónicos del EDA en las 2187 cosechas


### 2.2 Preprocesador completo

El preprocesador combina los dos transformadores de 2.1 con la imputación por mediana, las indicadoras de la tenencia y el escalado opcional. Se construye con una función porque se usa con y sin escalado, y con variables adicionales en las pruebas de robustez.

Se verifica con un ejemplo de seis cosechas construido a mano que la matriz de diseño tenga las 11 columnas esperadas, que no queden faltantes, que las medianas de `semsmad` y `dosismad` sean las del ejemplo, que el código 1 de la tenencia sea la referencia y que, con escalado, cada columna tenga media 0 y desviación 1.

In [4]:
# 2.2 Preprocesador completo: lluvias, numéricas, armónicos del mes, indicadoras de tenencia y escalado opcional
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CODIGOS_TENENCIA = [1, 2, 3]  # fijos de antemano: todos los pliegues producen las mismas indicadoras
NUMERICAS_MEDIANA = [columna for columna in PREDICTORES_NUMERICOS if columna != "lluvias"]


def crear_preprocesador(numericas_extra=(), escalar=True):
    """Imputación, codificación y escalado opcional. Todos los estadísticos se ajustan con los datos de fit().

    numericas_extra: variables numéricas adicionales, imputadas con la mediana (pruebas de robustez).
    """
    columnas = ColumnTransformer(
        [
            ("lluvias", ImputadorLluvias(), ["lluvias", PREDICTOR_MES]),
            ("numericas", SimpleImputer(strategy="median"), NUMERICAS_MEDIANA + list(numericas_extra)),
            ("mes", ArmonicosMes(), [PREDICTOR_MES]),
            ("tenencia", OneHotEncoder(categories=[CODIGOS_TENENCIA], drop="first", sparse_output=False),
             PREDICTORES_CATEGORICOS),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    pasos = [("columnas", columnas)]
    if escalar:
        pasos.append(("escalado", StandardScaler()))
    return Pipeline(pasos).set_output(transform="pandas")


# Verificación con un ejemplo de seis cosechas construido a mano
ejemplo = pd.DataFrame({
    "edad": [12.0, 13.5, 11.0, 14.0, 12.5, 13.0],
    "cortes": [1, 2, 3, 4, 2, 5],
    "semsmad": [8.0, np.nan, 10.0, 6.0, 9.0, 12.0],     # mediana de los observados: 9.0
    "lluvias": [100, np.nan, 300, np.nan, 2000, 150],   # el mismo ejemplo de 2.1
    "dosismad": [0.8, 1.0, np.nan, 1.2, 0.8, 1.0],      # mediana de los observados: 1.0
    "mes": [1, 1, 2, 3, 2, 12],
    "grupo_tenencia": [1, 2, 3, 3, 2, 1],
})
COLUMNAS_DISENO = ["lluvias", *NUMERICAS_MEDIANA, *columnas_armonicos, "grupo_tenencia_2", "grupo_tenencia_3"]

# Sin escalado: se pueden leer los valores imputados y las indicadoras
sin_escalar = crear_preprocesador(escalar=False).fit_transform(ejemplo)
assert list(sin_escalar.columns) == COLUMNAS_DISENO, list(sin_escalar.columns)
assert sin_escalar.notna().all().all(), "Quedaron faltantes"
assert sin_escalar.loc[1, "semsmad"] == 9.0 and sin_escalar.loc[2, "dosismad"] == 1.0, "Medianas incorrectas"
assert np.allclose(sin_escalar["lluvias"], lluvias_esperadas)
# El código 1 es la referencia: sus filas tienen 0 en las dos indicadoras
codigo_1 = ejemplo["grupo_tenencia"] == 1
assert (sin_escalar.loc[codigo_1, ["grupo_tenencia_2", "grupo_tenencia_3"]] == 0).all().all()

# Con escalado: media 0 y desviación 1 en cada columna del ajuste
escalado = crear_preprocesador().fit_transform(ejemplo)
assert np.allclose(escalado.mean(), 0) and np.allclose(escalado.std(ddof=0), 1), "El escalado no es el esperado"

# Variables adicionales para las pruebas de robustez: se agregan como columnas numéricas
con_extra = crear_preprocesador(numericas_extra=["pct_diatrea"], escalar=False).fit_transform(
    ejemplo.assign(pct_diatrea=[3.0, np.nan, 4.5, 2.0, 6.0, 5.0]))
assert "pct_diatrea" in con_extra.columns and con_extra.shape[1] == len(COLUMNAS_DISENO) + 1

print(f"Matriz de diseño: {len(COLUMNAS_DISENO)} columnas → {', '.join(COLUMNAS_DISENO)}")
display(sin_escalar.round(3))

Matriz de diseño: 11 columnas → lluvias, edad, cortes, semsmad, dosismad, mes_sin_12, mes_cos_12, mes_sin_6, mes_cos_6, grupo_tenencia_2, grupo_tenencia_3


,lluvias,edad,cortes,semsmad,dosismad,mes_sin_12,mes_cos_12,mes_sin_6,mes_cos_6,grupo_tenencia_2,grupo_tenencia_3
0,100.0,12.0,1.0,8.0,0.8,0.500,0.866,0.866,0.5,0.0,0.0
1,100.0,13.5,2.0,9.0,1.0,0.500,0.866,0.866,0.5,1.0,0.0
2,300.0,11.0,3.0,10.0,1.0,0.866,0.500,0.866,-0.5,0.0,1.0
3,225.0,14.0,4.0,6.0,1.2,1.000,0.000,0.000,-1.0,0.0,1.0
4,1949.0,12.5,2.0,9.0,0.8,0.866,0.500,0.866,-0.5,1.0,0.0
5,150.0,13.0,5.0,12.0,1.0,-0.000,1.000,-0.000,1.0,0.0,0.0


### Conclusión de la sección 2

Los transformadores reproducen los valores calculados a mano en el ejemplo:

- `ImputadorLluvias` reemplaza el faltante del mes 1 por la mediana de ese mes (100 mm). Usa la mediana general de los valores observados (225 mm) en el mes 3, que no tiene datos en el ajuste, y acota el valor extremo de 2000 mm en el percentil 99 de los observados (1949 mm). Los demás valores observados no cambian.
- `ArmonicosMes` coincide con los armónicos calculados en el EDA en las 2187 cosechas. Como es una transformación sin estado, calcularlos dentro del `Pipeline` no cambia los datos que recibe el modelo y permite tratar el mes como una sola variable al interpretar.

El preprocesador completo produce una matriz de diseño de 11 columnas, sin faltantes:

| Grupo | Columnas |
|---|---|
| Lluvia imputada y acotada | `lluvias` |
| Numéricas imputadas con la mediana | `edad`, `cortes`, `semsmad`, `dosismad` |
| Armónicos del mes | `mes_sin_12`, `mes_cos_12`, `mes_sin_6`, `mes_cos_6` |
| Indicadoras de tenencia (referencia: código 1) | `grupo_tenencia_2`, `grupo_tenencia_3` |

En el ejemplo, `semsmad` y `dosismad` se imputan con sus medianas (9.0 semanas y 1.0 L/ha), las cosechas con código de tenencia 1 tienen 0 en las dos indicadoras y, con escalado, cada columna queda con media 0 y desviación 1. El preprocesador también acepta variables adicionales para las pruebas de robustez.

Todos los estadísticos (medianas, tope de la lluvia, media y desviación del escalado) se calculan en `fit()`. Dentro de la validación cruzada se recalculan en cada pliegue con sus datos de ajuste, y el conjunto de prueba recibe los valores aprendidos en el entrenamiento. Las verificaciones con los datos reales se hacen en la sección 3, una vez definido el conjunto de entrenamiento.